# Run results, native analysis, OVITO, BPM, and PuMA export

`Recipe.run()` returns geometry plus scalar solver, topology, transfer,
event, checkpoint, and junction reports. Final geometry is downloaded
once; ordinary batches download only compact status values unless debug
snapshots were requested. Check `converged`, residuals, warnings, and
recipe events before claiming a relaxed specimen. Export methods do
not independently certify mechanical admissibility.

`run()` never modifies its inputs. The relaxed geometry is a new
`Assembly` at `result.assembly`, which can seed a follow-on `Recipe`.
A step that cannot meet its policy raises `tangle.RecipeError` instead
of returning a result (tutorial 09).

In [ ]:
# Export methods accept pathlib paths as well as strings.
from pathlib import Path
import tangle
from tangle.units import mm, um

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `fiber_count` | Fibers in the assembly. | count |
| `iterations` | Lifetime solver iteration. | count |
| `converged` | Whether the final acceptance criteria passed. | boolean |
| `max_penetration` | Largest final capsule overlap. | m |
| `max_curvature_ratio` | Largest final admissible-curvature utilization. | ratio |
| `active_segments` | Segments active after adaptation. | count |
| `active_vertices` | Vertices active after adaptation. | count |
| `segment_splits` | Accepted adaptive splits. | count |
| `segment_merges` | Accepted adaptive merges. | count |
| `refinement_passes` | Refinement evaluations. | count |
| `coarsening_passes` | Coarsening evaluations. | count |
| `uploaded_bytes` | Host-to-device transfer volume. | bytes |
| `downloaded_bytes` | Device-to-host transfer volume. | bytes |
| `cell_count` | Broad-phase cells in the final grid. | count |
| `events` | Human-readable recipe/solver events. | list of strings |
| `warnings` | Nonfatal numerical warnings. | list of strings |
| `junction_captures` | Reports from explicit capture operations. | list of strings |
| `resumed` | Whether this run loaded a checkpoint. | boolean |
| `resumed_iteration` | Iteration restored from a checkpoint. | count or `None` |
| `checkpoint_saves` | Checkpoints written by this invocation. | count |
| `last_checkpoint_iteration` | Iteration of the last save. | count or `None` |
| `last_checkpoint_bytes` | Serialized size of the last save. | bytes or `None` |
| `debug_ovito_frames` | Sparse debug trajectory frames written during the run. | count |
| `junction_count` | Persistent junctions in the result. | count |
| `assembly` | Relaxed geometry as a new assembly; the recipe's inputs are unchanged. | `Assembly` |

In [ ]:
# Build a minimal intersecting pair so the result contains meaningful
# contact and curvature diagnostics.
cell = tangle.Cell([1 * mm, 1 * mm, 1 * mm])
fibers = tangle.generate_fiber_pair_crossing(
    cell,
    material=tangle.Material("fiber", diameter=19 * um),
    length=0.8 * mm,
    axis_separation=10 * um,
)
recipe = tangle.Recipe(cell)
recipe.insert(fibers)
recipe.relax_until_converged(max_iterations=2_000)

# Keep reference notebooks safe to execute top-to-bottom by default.
RUN_SOLVER = False
if RUN_SOLVER:
    # Default tolerances suit millimeter-scale fibers, so state
    # micrometer ones here. The default backend is "wgpu"; add
    # backend="cpu" on a machine without a supported GPU.
    settings = tangle.RelaxationSettings(penetration_tolerance=0.1 * um, max_step=2 * um)
    try:
        result = recipe.run(settings)
    except tangle.RecipeError as error:
        # The error names the failing step and why it failed.
        print(error.operation_index, error.operation, error.iteration, error.reason)
        raise
    print(result.events)
    print(result.warnings)
    final_centerlines = result.centerlines()
    # The relaxed Assembly can seed a follow-on recipe.
    relaxed = result.assembly
    follow_on = tangle.Recipe(relaxed)
    print(relaxed.fiber_count, relaxed.cell.lengths)

## OVITO output

`coloring` accepts `"fiber"`, `"curvature_ratio"`, or
`"refinement_level"`. The optional view script and OVITO session make
oriented spherocylinder rendering reproducible.

In [ ]:
if RUN_SOLVER:
    # The generated view script restores oriented spherocylinder shape
    # and coloring when the raw dump is opened in OVITO.
    output = Path("output")
    output.mkdir(exist_ok=True)
    result.write_ovito(
        output / "result.dump",
        view_script_path=output / "result_view.py",
        session_path=output / "result.ovito",
        coloring="curvature_ratio",
    )

## BPM export

`export_bpm` writes bonded spheres or bonded spherocylinders without
owning a downstream solver's runtime configuration or constitutive
laws. The four modes separate exact centerline sampling from
endpoint-preserving or constant-resolution sampling:

- `spheres_exact` keeps the requested arc spacing exactly and shares
  unused length between the two fiber ends.
- `spheres_dynamic` preserves every source segment endpoint and adjusts
  the spacing inside each segment.
- `spherocylinders_exact` writes one capsule per active segment.
- `spherocylinders_constant` uses the shortest active segment length
  throughout, with a shorter remainder only where required.

Sphere center spacing is specified in radii and must lie from `1/3`
through `1`. `density`, `atom_type`, and `bond_type` provide the
material and topology metadata required by BPM consumers such as DIRT.

In [ ]:
if RUN_SOLVER:
    # Dynamic spheres retain every TANGLE segment endpoint; actual bond
    # spacing may shift locally around the requested one-third radius.
    particles, bonds = result.export_bpm(
        output / "result.data",
        mode="spheres_dynamic",
        sphere_spacing_over_radius=1.0 / 3.0,
        density=1800.0,
        atom_type=1,
        bond_type=1,
    )
    print(particles, bonds)

## Native characterization

`characterize()` calls the Rust `tangle_characterize` crate on the
exact centerlines and sections; `result.assembly.characterize()` gives
the same report. It reports nominal swept volume rather than the
geometric union of solids, so the field is deliberately named
`nominal_swept_volume_fraction`. Curvature is summarized by
`max_curvature`, `max_curvature_ratio`, and
`curvature_limit_violations`. The complete versioned report is
available as a dictionary, JSON string, or JSON file.

In [ ]:
if RUN_SOLVER:
    # Length weighting treats centerline equally; volume weighting is
    # the appropriate voxel comparison when diameters differ.
    analysis = result.characterize()
    print(analysis.length_weighted_orientation_tensor)
    print(analysis.volume_weighted_orientation_tensor)
    print(analysis.max_curvature_ratio, analysis.curvature_limit_violations)
    analysis.write_json(output / "tangle_analysis.json")

## PuMA-compatible voxel bundle

`export_puma()` also dispatches to Rust. It writes cell-centered phase
IDs and exact tangents to `domain.vti`, optional fiber-ID and smooth
interface images, a manifest, and native analysis JSON. PuMA stays an
independent optional application: import `pumapy` directly in the
analysis environment rather than through a TANGLE adapter. The current
voxelizer runs on the host and supports circular sections in diagonal
orthorhombic cells. It validates assembly structure and grid settings,
not residual penetration or final solver acceptance.

In [ ]:
if RUN_SOLVER:
    # The cubic voxel size must tile all three orthorhombic cell edges.
    puma_bundle = result.export_puma(
        output / "result.puma",
        voxel_size=20 * um,
        include_fiber_ids=True,
        include_interface=True,
    )
    print(puma_bundle.domain_path)
    print(puma_bundle.voxel_volume_fraction)
    print(puma_bundle.ambiguous_voxels)